In [1]:
# 1) ensure project root is importable
import sys
import os
import os

from pathlib import Path

# Get the current folder path
current_folder = Path.cwd()
print("Current:", current_folder)

# Get the path of the folder one level up
parent_folder = current_folder.parent
print("Parent:", parent_folder)

# If you actually need to change your working directory to it:
import os
os.chdir(parent_folder)

# 2) imports
from sentence_transformers import SentenceTransformer
from src.config import CFG
from src.vector_db import collection
from src.generator import get_llm
from src.retriever import MongoHybridRetriever

# 3) build components
embedder = SentenceTransformer(CFG["embedding"]["model"])
llm = get_llm()

# 4) instantiate retriever
retriever = MongoHybridRetriever(
    embedder=embedder,
    llm=llm,
    collection=collection,
    use_mmr=True
)

# 5) call it (either .invoke or .get_relevant_documents)
docs = retriever.invoke("What employee expenses has JUSNL projected?")
# docs is a list of LangChain Document objects
for d in docs:
    print(d.metadata.get("_id"), d.page_content[:300])


Current: d:\python scripts\Generative AI\arise_chatbot\research
Parent: d:\python scripts\Generative AI\arise_chatbot


d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-23 21:29:56,267 INFO arise.pipeline: Query rewrite strategy configured: 
2026-06-23 21:29:56,268 INFO arise.pipeline: hybrid_retrieve.start


What employee expenses has JUSNL projected? : Before
What employee expenses has JUSNL projected? : After
multi:


2026-06-23 21:29:57,819 INFO arise.pipeline: hybrid_retrieve.results
2026-06-23 21:29:57,821 INFO arise.pipeline: hybrid_retrieve.fused
2026-06-23 21:29:57,825 INFO arise.pipeline: mmr.start
2026-06-23 21:29:57,830 INFO arise.pipeline: mmr.selected


6a270f3513f34b0acd35eaf8 [Document: JUSNL | Section: 5.5. Operation and Maintenance Expenses | Paragraphs: 5.5.5]

Annual Performance Review exercise and true up the employee cost and A&G
expenses on account of this variation, for the Control Period;
Note 2: Any variation due to changes recommended by the Pay Commission or

6a270f3513f34b0acd35eacb [Document: JUSNL | Section: 4.5. Operation and Maintenance Expenses | Paragraphs: 4.5.1, 4.5.2]

4.5. Operation and Maintenance Expenses
4.5.1. The O&M expenses of JUSNL for the FY 2024-25 have been projected
considering the historical expenses and the projections in terms of capitalization
etc. The
6a270f3513f34b0acd35eaff Table 1:
  Row 1 → FY 2025-26: Approved in T.O  dated 23.06.2023 | FY 2025-26: Projected
  Row 2 → Particulars: Employee Cost | FY 2025-26: 60.89 | FY 2025-26: 107.54
  Row 3 → Particulars: R&M | FY 2025-26: 59.04 | FY 2025-26: 107.46
  Row 4 → Particulars: A&G | FY 2025-26: 11.38 | FY 2025-26: 26.9
6a270f3513f34b0acd35ea

In [18]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [21]:
# 1. Import the compression retriever from langchain-classic
from langchain_classic.retrievers import ContextualCompressionRetriever

# 2. Import your chosen compressor from langchain-community or langchain-classic
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor
# OR if using another compressor (e.g., LLMLingua):
# from langchain_community.document_compressors import LLMLinguaCompressor

# 3. Setup your workflow

compressor = LLMChainExtractor.from_llm(llm)

# 4. Wrap your base vector store retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)


compressed_docs = compression_retriever.invoke(
    "What employee expenses has JUSNL projected?"
)
pretty_print_docs(compressed_docs)

2026-06-23 21:40:08,539 INFO arise.pipeline: Query rewrite strategy configured: 
2026-06-23 21:40:08,539 INFO arise.pipeline: hybrid_retrieve.start


What employee expenses has JUSNL projected? : Before
What employee expenses has JUSNL projected? : After
multi:


2026-06-23 21:40:09,341 INFO arise.pipeline: hybrid_retrieve.results
2026-06-23 21:40:09,342 INFO arise.pipeline: hybrid_retrieve.fused
2026-06-23 21:40:09,342 INFO arise.pipeline: mmr.start
2026-06-23 21:40:09,345 INFO arise.pipeline: mmr.selected


Document 1:

[Document: JUSNL | Section: 5.5. Operation and Maintenance Expenses | Paragraphs: 5.5.5]

5.5.5. The Petitioner has projected the employee cost for the FY 2025-26 by escalating
the projected employee cost (excluding the terminal benefits) estimated for FY
2024-25 by the inflation factor of 6.09%. The same has been approved by the
Hon’ble Commission in the MYT Order for the 3rd Control Period. Also, it is
submitted that JUSNL proposes to induct 96 junior managers and 10 managers
during the FY 2025-26. The estimated salary of the proposed new employees is
projected at Rs. 8.38 Crores.
----------------------------------------------------------------------------------------------------
Document 2:

[Document: JUSNL | Section: 4.5. Operation and Maintenance Expenses | Paragraphs: 4.5.2]

4.5.2. Operation and Maintenance expenses comprise of the following heads:
 Employees Expenses which includes the salaries, dearness allowances,
dearness pay, other allowances, incentives and 

In [22]:
original_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(docs)]))
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

print("Original context length:", original_contexts_len)
print("Compressed context length:", compressed_contexts_len)
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")


Original context length: 16831
Compressed context length: 5583
Compressed Ratio: 3.01x
